In [43]:
'''# How to run this code in Google Colab or Local Machine: 
#Before running this code, make sure to install the required python and needed packages. 
# Since Iam used to Visual studio code, below instructions are for VSC but can be true fot other similar IDEs.
# If you don't want to install them in Local Machine then you use the colab directly. 
# Choose Runtime -> Python 3(ipykernel)
# If the above is not available, Select another kernel -> Colab -> Auto Conect or New Colab server if you do not have

'''

import sys
from datetime import datetime
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Runtime is running Date and time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))



''' Here we are using the Gdrive to store the file and read it from there. 
If you want to use the local machine or colab folder, you can use the commented code below to read the file.
'''
# If the file is stored in the local machine or in Colab folder, you can use the following code to read the file.
# file_path = r"C:\Users\veera\AI\SampleFile\test-product-review.parquet"
# file_path = r"\sample_data\test-product-review.parquet"

from google.colab import drive
import os
drive.mount('/content/drive')
print("Drive mounted:", os.path.ismount('/content/drive'))

file_path = '/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet'
print("File exists:"+file_path+" -", os.path.isfile(file_path))



Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Executable: /usr/bin/python3
Runtime is running Date and time: 2026-10-30 23:06:57
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted: True
File exists:/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet - True


In [47]:
import pandas as pd
import pandas as pd
import pyarrow as pa

import pyarrow.parquet as pq
from tqdm import tqdm

# Load data from Parquet (removed the 'en' filter to allow multilingual reviews)
table = pq.read_table(
    file_path, columns=["review_id", "review_body", "language", "stars"]
)
df = table.to_pandas()

!pip install -q deep-translator



In [49]:
from deep_translator import MyMemoryTranslator, exceptions
import pandas as pd
from tqdm import tqdm
import time

translated_reviews = []

for text, lang in tqdm(
    zip(df["review_body"], df["language"]),
    total=len(df),
    desc="Translating with MyMemory",
):
  if pd.isna(text) or str(text).strip() == "":
    translated_reviews.append("")
  elif lang == "en":
    # Skip English reviews
    print(f"Skipping translation for English review: {text}")
    translated_reviews.append(text)
  else:
    try:
      # Use the specific language code from your dataframe as the source
      translator = MyMemoryTranslator(source=lang, target="en")
      translated = translator.translate(str(text))
      translated_reviews.append(translated if translated else text)
      time.sleep(0.1)  # Gentle pause to prevent rate limits
      print(f"Translated ({lang}): {text} -> {translated}")
    except (exceptions.LanguageNotSupportedException, Exception):
      # If MyMemory doesn't support that specific language code, keep original text
      translated_reviews.append(text)
      print(f"Translation failed for: {text}, keeping original.")

df["review_body_Translated"] = translated_reviews

print("Translation complete!")
print(
    df[["review_body", "review_body_Translated", "language", "stars"]].head()
)

Translating with MyMemory: 100%|██████████| 1500/1500 [00:00<00:00, 15244.36it/s]

Translation failed for: Bon produit me renvoyer car cela taille petit, keeping original.
Skipping translation for English review: Well I haven't stunned anyone...yet. So I can only rate what I know. I bought this because it was small and I could attach it to my belt. The belt loop strap on the case broke in the first 4 hours I had it. So now the convienence of being able to carry it on my side is lost. Now I'm going to have to invest in a case to carry it in.
Translation failed for: Lieferung super schnell allerdings tat sich nachdem anschließen des Mini PC gar nichts. Egal ob über hdmi oder VGA Kabel angeschlossen. Monitor blieb dunkel. Auch mit einem neuen hdmi Kabel tat sich. Rückgabe habe ich umgehend eingeleitet. Mal sehen ob ich dem Teil nochmal eine Chance geben., keeping original.
Translation failed for: Farbe wie beschrieben und die Größe passt auch wie angegeben., keeping original.
Translation failed for: Akku lässt sich leicht gegen den Originalen tauschen, aber beim ersten 